In [45]:
!apt-get install -y gdal-bin libgdal-dev 2>/dev/null | tail -3
!pip install -q rasterio==1.3.10 shapely psycopg2-binary boto3 numpy tqdm
!pip install boto3 botocore

#### Data Transformation
!pip install -q pyspark apache-sedona
from google.colab import userdata
AWS_ACCESS_KEY_ID     = userdata.get('AWS_ACCESS_KEY_ID')
AWS_SECRET_ACCESS_KEY = userdata.get('AWS_SECRET_ACCESS_KEY')
print('AWS credentials loaded.')
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, month, year, avg

spark = SparkSession.builder \
    .appName("VegetationPipeline") \
    .config("spark.driver.memory", "4g") \
    .config("spark.hadoop.fs.s3a.access.key", AWS_ACCESS_KEY_ID) \
    .config("spark.hadoop.fs.s3a.secret.key", AWS_SECRET_ACCESS_KEY) \
    .config("spark.hadoop.fs.s3a.endpoint", "s3.amazonaws.com") \
    .getOrCreate()
print(f"Spark initialized: {spark.version}")
# Tile N39W108 covers lat 39-42°N, lon -108 to -105°W (includes Boulder County)
!curl -L https://esa-worldcover.s3.eu-central-1.amazonaws.com/v100/2020/map/ESA_WorldCover_10m_2020_v100_N39W108_Map.tif -o /tmp/worldcover_boulder.tif
print('WorldCover tile downloaded to /tmp/worldcover_boulder.tif')
import numpy as np
import boto3
import rasterio
from rasterio.session import AWSSession
from rasterio.transform import Affine
import re

def extract_scene_date(tile_id, file_url):
    """
    Extract scene date from file_url/tile_id patterns.
    Supported patterns:
      1) veg_YYYY-MM-DD
      2) HLS.<prod>.<tile>.YYYYDOY...
      3) tile_id suffix _YYYYDDD
    Returns datetime.date or None.
    """
    url = str(file_url)
    tid = str(tile_id)

    m = re.search(r'veg_([0-9]{4})-([0-9]{2})-([0-9]{2})', url)
    if m:
        y, mo, d = int(m.group(1)), int(m.group(2)), int(m.group(3))
        return datetime.date(y, mo, d)

    m = re.search(r'HLS\.[SL]30\.[A-Z0-9]+\.([0-9]{4})([0-9]{3})T', url)
    if m:
        y, doy = int(m.group(1)), int(m.group(2))
        return datetime.date(y, 1, 1) + datetime.timedelta(days=doy - 1)

    m = re.search(r'_([0-9]{7})$', tid)
    if m:
        y = int(m.group(1)[:4])
        doy = int(m.group(1)[4:])
        return datetime.date(y, 1, 1) + datetime.timedelta(days=doy - 1)

    return None

def strip_date_from_tile_id(tile_id):
    """
    Convert date-stamped tile ids like veg_2017-10-01 into stable ids like veg.
    Leave non-matching ids unchanged.
    """
    return re.sub(r'_[0-9]{4}-[0-9]{2}-[0-9]{2}$', '', str(tile_id))

def read_bands_from_s3(results, aws_key, aws_secret, region='us-west-1',
                        red_band=1, nir_band=2, swir1_band=3, pixel_stride=10):
    """
    Reads Red, NIR, and SWIR1 band arrays directly from S3 COGs for each
    scene in results. Streams via rasterio AWSSession — no local download.

    Returns list of (tile_id, parsed_date, red, nir, swir1, bbox_wkt,
                      strided_transform, crs_str) per scene.
    strided_transform and crs_str are needed by apply_forest_mask.
    """
    boto3_session = boto3.Session(
        aws_access_key_id=aws_key,
        aws_secret_access_key=aws_secret,
        region_name=region,
    )
    aws_session = AWSSession(boto3_session)
    scene_bands = []
    for tile_id, _db_acq_date, file_url, _, _res, bbox_wkt in results:
        parsed_date = extract_scene_date(tile_id, file_url)
        if parsed_date is None:
            print(f'  [WARN] Could not parse date from path/id. Skipping scene: {tile_id} -> {file_url}')
            continue
        print(f'  Reading {tile_id}  ->  {file_url}')
        print(f'    parsed_date={parsed_date}')
        with rasterio.Env(aws_session):
            with rasterio.open(file_url) as src:
                n = src.count
                print(f'    bands={n}  shape={src.height}x{src.width}  crs={src.crs}')
                rb = min(red_band,   n)
                nb = min(nir_band,   n)
                sb = min(swir1_band, n)
                red   = src.read(rb)[::pixel_stride, ::pixel_stride].astype('float32')
                nir   = src.read(nb)[::pixel_stride, ::pixel_stride].astype('float32')
                swir1 = src.read(sb)[::pixel_stride, ::pixel_stride].astype('float32')
                # Use rasterio's actual dataset mask — 255=valid, 0=nodata/masked
                ds_mask = src.dataset_mask()[::pixel_stride, ::pixel_stride]
                invalid = ds_mask == 0
                for arr in (red, nir, swir1):
                    arr[invalid] = np.nan
                # Adjust affine transform for the stride so apply_forest_mask
                # can reproject WorldCover onto this exact subsampled grid
                t = src.transform
                strided_transform = Affine(
                    t.a * pixel_stride, t.b, t.c,
                    t.d, t.e * pixel_stride, t.f,
                )
                crs_str = src.crs
        print(f'    red   [{np.nanmin(red):.0f}, {np.nanmax(red):.0f}]')
        print(f'    nir   [{np.nanmin(nir):.0f}, {np.nanmax(nir):.0f}]')
        print(f'    swir1 [{np.nanmin(swir1):.0f}, {np.nanmax(swir1):.0f}]')
        scene_bands.append((tile_id, parsed_date, red, nir, swir1, bbox_wkt,
                             strided_transform, crs_str))
    return scene_bands

import datetime
import psycopg2
from psycopg2 import sql

# Redefine necessary variables for CONFIG, including userdata calls
from google.colab import userdata
import os

AWS_ACCESS_KEY_ID     = userdata.get('AWS_ACCESS_KEY_ID')
AWS_SECRET_ACCESS_KEY = userdata.get('AWS_SECRET_ACCESS_KEY')
AWS_REGION            = 'us-west-1'
S3_BUCKET_NAME        = userdata.get('S3_BUCKET_NAME')

DB_CONFIG = {
    'host':     userdata.get('DB_HOST'),
    'port':     5432,
    'database': 'postgres',
    'user':     'postgres',
    'password': userdata.get('DB_PASSWORD'),
    'connect_timeout': 10
}
S3_RAW_PREFIX = 'raw/'

LOCAL_WORK_DIR = '/tmp/vegetation_pipeline'
os.makedirs(LOCAL_WORK_DIR, exist_ok=True)

# Placeholder for DRIVE_SOURCE_DIR, as it's not strictly needed for this cell's function
# and is commented out in its original cell.
DRIVE_SOURCE_DIR = '/content/drive/MyDrive/BigDataProject' # Assuming this default path if needed elsewhere

# Redefine CONFIG dictionary
CONFIG = {
    'aws_access_key_id':     AWS_ACCESS_KEY_ID,
    'aws_secret_access_key': AWS_SECRET_ACCESS_KEY,
    'aws_region':            AWS_REGION,
    's3_bucket':             S3_BUCKET_NAME,
    's3_cog_prefix':         'cog/',
    's3_raw_prefix':         S3_RAW_PREFIX,
    'db_config':             DB_CONFIG,
    'phase':                 1, # Defaulting to phase 1, though not directly used here.
    'drive_source_dir':      DRIVE_SOURCE_DIR,
    'local_work_dir':        LOCAL_WORK_DIR,
    'upload_to_s3':          True,
    'skip_existing':         True,
    'cog_compress':          'LZW',
    'cog_blocksize':         512,
    'cog_overviews':         [2, 4, 8, 16],
    'compute_ndvi_preview':  True,
    'ndvi_sample_size':      1000,
}

def get_db_connection(config):
    db_config = config['db_config']
    try:
        conn = psycopg2.connect(
            host=db_config['host'],
            port=db_config['port'],
            database=db_config['database'],
            user=db_config['user'],
            password=db_config['password'],
            connect_timeout=db_config.get('connect_timeout', 10)
        )
        conn.autocommit = False   # Use explicit transactions

        # Test PostGIS availability
        cur = conn.cursor()
        cur.execute("CREATE EXTENSION IF NOT EXISTS postgis;")
        cur.execute("SELECT PostGIS_Version();")
        postgis_ver = cur.fetchone()[0]
        cur.execute("SELECT version();")
        pg_ver = cur.fetchone()[0].split(' ')[1]

        print(f"  Database connected successfully!")
        print(f"  PostgreSQL version: {pg_ver}")
        print(f"  PostGIS version: {postgis_ver.split(' ')[0]}")
        cur.close()
        return conn

    except psycopg2.OperationalError as e:
        print(f"  [ERROR] Cannot connect to database: {e}")
        print(f"  Check: 1) RDS endpoint correct? 2) Security group allows port 5432?")
        print(f"         3) RDS instance is 'Available' in AWS console?")
        return None

def run_spatiotemporal_query(conn, west, south, east, north,
                              start_date, end_date):
    cur = conn.cursor()
    cur.execute("""
        SELECT
            tile_id,
            acquisition_date,
            file_url,
            ndvi_mean,
            resolution_m,
            ST_AsText(geom) AS bbox_wkt
        FROM vegetation_metadata
        WHERE
            ST_Intersects(
                geom,
                ST_MakeEnvelope(%s, %s, %s, %s, 4326)
            )
            AND acquisition_date BETWEEN %s AND %s
        ORDER BY acquisition_date;
    """, (west, south, east, north, start_date, end_date))

    results = cur.fetchall()
    cur.close()
    return results

# Establish DB connection using the CONFIG dictionary defined earlier
conn_transform = get_db_connection(config=CONFIG)
if conn_transform:
    # Query for all COG URLs. Use a very wide spatial and temporal range
    # to capture all indexed scenes.
    all_bounds = (-180.0, -90.0, 180.0, 90.0) # Global extent
    all_start_date = datetime.date(1900, 1, 1) # Start of modern remote sensing
    all_end_date   = datetime.date.today() + datetime.timedelta(days=365) # Extend to future proof

    print("Fetching all COG URLs from PostGIS database...")
    results = run_spatiotemporal_query(
        conn_transform,
        *all_bounds,
        all_start_date, all_end_date
    )
    conn_transform.close()
    print(f'Found {len(results)} scene(s) in the database.')
else:
    print('Failed to connect to database. Cannot fetch scene URLs.')
    results = [] # Ensure results is an empty list if DB connection fails

if results:
    scene_bands = read_bands_from_s3(
        results,
        AWS_ACCESS_KEY_ID,
        AWS_SECRET_ACCESS_KEY,
        region=AWS_REGION,
        red_band=1, nir_band=2, swir1_band=3,  # Adjusted band assignments
        pixel_stride=10,   # set to 1 for full resolution
    )
    print(f'Bands loaded from S3 for {len(scene_bands)} scene(s). No download needed.')
else:
    print('No results found or database connection failed. Skipping band reading.')
    scene_bands = [] # Ensure scene_bands is defined.
def build_pixel_dataframe(sedona, scene_bands):
    """
    Flattens per-scene band arrays into a Spark DataFrame with one row per pixel.
    Columns: tile_id, date, red, nir, swir1, pixel_id, true_pixel_id, bbox_wkt, lon, lat
    NaN pixels (incl. non-forest pixels zeroed by apply_forest_mask) are dropped.
    lon/lat are the pixel centroid coordinates derived from the strided_transform (index 6).
    """
    pixel_rows = []
    for scene in scene_bands:
        tile_id, acq_date, red, nir, swir1, bbox_wkt = scene[:6]
        transform = scene[6] if len(scene) > 6 else None  # strided affine transform
        true_tile_id = strip_date_from_tile_id(tile_id)
        h, w = red.shape
        for r in range(h):
            for c in range(w):
                rv, nv, sv = float(red[r, c]), float(nir[r, c]), float(swir1[r, c])
                if rv != rv or nv != nv or sv != sv:  # NaN check
                    continue
                # Compute pixel centroid lon/lat from the affine transform.
                # transform * (col, row) = (lon, lat) using rasterio convention.
                if transform is not None:
                    lon = float(transform.c + c * transform.a)
                    lat = float(transform.f + r * transform.e)
                else:
                    lon, lat = None, None
                pixel_rows.append((
                    tile_id, acq_date,
                    rv, nv, sv,
                    f'{tile_id}_{r}_{c}',
                    f'{true_tile_id}_{r}_{c}',
                    bbox_wkt, lon, lat,
                ))
    print(f'  Pixel rows (non-NaN): {len(pixel_rows):,}')
    return sedona.createDataFrame(
        pixel_rows,
        ['tile_id', 'date', 'red', 'nir', 'swir1', 'pixel_id', 'true_pixel_id', 'bbox_wkt', 'lon', 'lat'],
    )

from rasterio.warp import reproject, Resampling

def apply_forest_mask(scene_bands, worldcover_path, forest_class=10):
    """
    Masks each scene's band arrays to forested pixels only using ESA WorldCover.
    Class 10 = Tree cover. Non-forest pixels are set to NaN.

    Operates on numpy arrays (before Spark) by reprojecting the WorldCover
    GeoTIFF onto each scene's subsampled pixel grid using rasterio.
    No Sedona, no geoparquet, no geom column required.

    Parameters
    ----------
    scene_bands     : list of 8-tuples from read_bands_from_s3()
                      (..., strided_transform, crs_str)
    worldcover_path : local path to the ESA WorldCover GeoTIFF
    forest_class    : WorldCover class for tree cover (default 10)
    """
    masked = []
    with rasterio.open(worldcover_path) as wc:
        for scene in scene_bands:
            tile_id, acq_date, red, nir, swir1, bbox_wkt, transform, crs = scene
            red, nir, swir1 = red.copy(), nir.copy(), swir1.copy()  # avoid in-place mutation of scene_bands
            h, w = red.shape
            forest_mask = np.zeros((h, w), dtype=np.uint8)
            reproject(
                source=rasterio.band(wc, 1),
                destination=forest_mask,
                dst_transform=transform,
                dst_crs=crs,
                resampling=Resampling.nearest,
            )
            is_forest = (forest_mask == forest_class)
            for arr in (red, nir, swir1):
                arr[~is_forest] = np.nan
            n_forest = int(is_forest.sum())
            print(f'  [{tile_id}] Forest pixels: {n_forest:,} / {h*w:,} '
                  f'({100*n_forest/max(h*w,1):.1f}%)')
            masked.append((tile_id, acq_date, red, nir, swir1, bbox_wkt, transform, crs))
    return masked

def compute_vegetation_indices(df):
    """
    Computes NDVI and NDMI from Red, NIR, and SWIR1 bands.
    NDVI = (NIR - Red)  / (NIR + Red)
    NDMI = (NIR - SWIR1) / (NIR + SWIR1)
    """
    return (
        df
        .withColumn('ndvi',
            when((col('nir') + col('red')) != 0,
                 (col('nir') - col('red')) / (col('nir') + col('red')))
            .otherwise(None))
        .withColumn('ndmi',
            when((col('nir') + col('swir1')) != 0,
                 (col('nir') - col('swir1')) / (col('nir') + col('swir1')))
            .otherwise(None))
    )

def harmonize_time_series(df):
    """
    Aggregates irregular observations into monthly composites.
    Returns a regular time series for each stable pixel.
    """
    return (
        df.groupBy('true_pixel_id', year('date').alias('year'), month('date').alias('month'))
          .agg(avg('ndvi').alias('ndvi_mean'), avg('ndmi').alias('ndmi_mean'))
          .dropna(subset=['ndvi_mean', 'ndmi_mean'], how='all')
          .orderBy('year', 'month')
    )

if 'scene_bands' in locals() and scene_bands:
    forest_data_path = '/tmp/worldcover_boulder.tif'
    masked_bands = apply_forest_mask(scene_bands, forest_data_path)
    spark_df   = build_pixel_dataframe(spark, masked_bands)
    indices_df = compute_vegetation_indices(spark_df)
    final_df   = harmonize_time_series(indices_df)
    final_df.show(10)
    print(f'Transformation Complete: {final_df.count()} monthly records generated.')
else:
    print('No scene_bands found. Run the S3 band-read cell above first.')
    final_df = None

# ── Timeframe Configuration ──────────────────────────────────────────────
# Set the evaluation window. All data BEFORE EVAL_START is used as the baseline.
# Format: 'YYYY-MM'. Set EVAL_END to None to include all data after EVAL_START.
EVAL_START = '2025-08'   # inclusive start of anomaly evaluation window
EVAL_END   = None   # inclusive end  (set to None for no upper bound)
MIN_BASELINE_MONTHS = 12  # minimum months of baseline required

# Z-Score Anomaly Detection
from pyspark.sql.functions import stddev_pop as stddev, mean as spark_mean, when, col
import pyspark.sql.functions as F

if final_df is None:
    raise ValueError('final_df is not defined. Upstream transformation did not produce any rows.')

def parse_ym(ym_str):
    y, m = ym_str.split('-')
    return int(y), int(m)

eval_start_year, eval_start_month = parse_ym(EVAL_START)
start_ym_int = eval_start_year * 100 + eval_start_month

# ── Validate: enough baseline history ────────────────────────────────────
data_min = final_df.agg(F.min(col('year') * 100 + col('month'))).collect()[0][0]
data_start_year, data_start_month = data_min // 100, data_min % 100
baseline_months = (eval_start_year - data_start_year) * 12 + (eval_start_month - data_start_month)
if baseline_months < MIN_BASELINE_MONTHS:
    raise ValueError(
        f'Insufficient baseline: only {baseline_months} month(s) before EVAL_START={EVAL_START}. '
        f'Need at least {MIN_BASELINE_MONTHS}. Move EVAL_START later or ingest more historical scenes.'
    )
print(f'Baseline window: {data_start_year}-{data_start_month:02d} to {EVAL_START} ({baseline_months} months)')

# ── Baseline: all data strictly before EVAL_START ────────────────────────
baseline_df = (
    final_df.filter((col('year') * 100 + col('month')) < start_ym_int)
    .groupBy('true_pixel_id', 'month')
    .agg(
        spark_mean('ndvi_mean').alias('ndvi_mu'),
        stddev('ndvi_mean').alias('ndvi_sigma'),
        spark_mean('ndmi_mean').alias('ndmi_mu'),
        stddev('ndmi_mean').alias('ndmi_sigma'),
    )
)

# ── Evaluation window ────────────────────────────────────────────────────
if EVAL_END:
    eval_end_year, eval_end_month = parse_ym(EVAL_END)
    end_ym_int = eval_end_year * 100 + eval_end_month
    eval_filter = (col('year') * 100 + col('month')).between(start_ym_int, end_ym_int)
else:
    eval_filter = (col('year') * 100 + col('month')) >= start_ym_int
eval_df = final_df.filter(eval_filter)
print(f'Evaluation window: {EVAL_START} to {EVAL_END or "end"}  ({eval_df.count()} pixel-months)')

# ── Join with baseline and compute Z-scores ──────────────────────────────
anomaly_df = eval_df.join(baseline_df, on=['true_pixel_id', 'month'])

anomaly_df = anomaly_df.withColumn(
    'ndvi_zscore',
    when(col('ndvi_sigma') > 0,
         (col('ndvi_mean') - col('ndvi_mu')) / col('ndvi_sigma'))
    .otherwise(0.0)
).withColumn(
    'ndmi_zscore',
    when(col('ndmi_sigma') > 0,
         (col('ndmi_mean') - col('ndmi_mu')) / col('ndmi_sigma'))
    .otherwise(0.0)
)

# Flag anomalies (Z < -2.0)
THRESHOLD = -1.5
anomaly_df = anomaly_df.withColumn(
    'is_ndvi_anomaly', col('ndvi_zscore') < THRESHOLD
).withColumn(
    'is_ndmi_anomaly', col('ndmi_zscore') < THRESHOLD
)

# Show flagged anomalies
anomalies = anomaly_df.filter(col('is_ndvi_anomaly') | col('is_ndmi_anomaly'))
anomalies.show(20)
print(f'Total anomalies detected: {anomalies.count()}')
print(f'Total records evaluated: {anomaly_df.count()}')

gdal-bin is already the newest version (3.8.4+dfsg-1~jammy0).
libgdal-dev is already the newest version (3.8.4+dfsg-1~jammy0).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
AWS credentials loaded.
Spark initialized: 4.0.2
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 85.2M  100 85.2M    0     0  13.5M      0  0:00:06  0:00:06 --:--:-- 17.2M
WorldCover tile downloaded to /tmp/worldcover_boulder.tif
  Database connected successfully!
  PostgreSQL version: 17.6
  PostGIS version: 3.5
Fetching all COG URLs from PostGIS database...
Found 157 scene(s) in the database.
  Reading veg_2015-08-09  ->  s3://vegetation-anomaly-cogs/cog/2015/veg_2015-08-09.cog.tif
    parsed_date=2015-08-09
    bands=3  shape=1073x1714  crs=EPSG:4326
    red   [3, 5470]
    nir   [2, 6087]
    swir1 [15, 6667]
  Reading veg_2015-12-06  ->  s3://vegetation-anomaly-cogs/cog/2015/veg_201

In [ ]:
# ── Timelapse Generator ─────────────────────────────────────────────────
# Generates 3 GIFs: NDVI heatmap, NDMI heatmap, and false-color NRG composite.
# Runs on masked_bands (forest-only, already loaded). Requires imageio.
!pip install -q imageio
import imageio, io
import numpy as np
import matplotlib.pyplot as plt

def generate_timelapse(scene_bands, output_path='/tmp/timelapse.gif',
                        mode='ndvi', fps=3, start_ym=None, end_ym=None):
    """
    Creates a GIF timelapse from scene_bands.

    mode: 'ndvi'        — NDVI heatmap  (green=healthy, red=stressed)
          'ndmi'        — NDMI heatmap  (blue=moist, yellow=dry)
          'false_color' — NRG composite (NIR→R, Red→G, SWIR1→B;
                           vegetation=magenta, bare soil=greenish)
    start_ym / end_ym : 'YYYY-MM' strings to filter scenes (inclusive).
    """
    scenes = sorted(scene_bands, key=lambda s: s[1])  # sort by acq_date
    if start_ym:
        sy, sm = (int(x) for x in start_ym.split('-'))
        scenes = [s for s in scenes if (s[1].year, s[1].month) >= (sy, sm)]
    if end_ym:
        ey, em = (int(x) for x in end_ym.split('-'))
        scenes = [s for s in scenes if (s[1].year, s[1].month) <= (ey, em)]
    if not scenes:
        raise ValueError(f'No scenes in timeframe [{start_ym}, {end_ym}].')

    def _norm(arr):
        lo, hi = np.nanpercentile(arr, 2), np.nanpercentile(arr, 98)
        return np.clip((arr - lo) / max(hi - lo, 1e-6), 0, 1)

    def _index(a, b):
        with np.errstate(invalid='ignore', divide='ignore'):
            return np.where((a + b) > 0, (a - b) / (a + b), np.nan)

    frames = []
    for tile_id, acq_date, red, nir, swir1, *_ in scenes:
        fig, ax = plt.subplots(figsize=(6, 4), dpi=80)
        ax.set_title(f'{acq_date}', fontsize=10, fontweight='bold')
        ax.axis('off')
        if mode == 'ndvi':
            img = ax.imshow(_index(nir, red), cmap='RdYlGn', vmin=-0.2, vmax=0.8)
            plt.colorbar(img, ax=ax, fraction=0.03, pad=0.02, label='NDVI')
        elif mode == 'ndmi':
            img = ax.imshow(_index(nir, swir1), cmap='RdBu', vmin=-0.5, vmax=0.5)
            plt.colorbar(img, ax=ax, fraction=0.03, pad=0.02, label='NDMI')
        elif mode == 'false_color':
            rgb = np.nan_to_num(np.dstack([_norm(nir), _norm(red), _norm(swir1)]))
            ax.imshow(rgb)
        buf = io.BytesIO()
        plt.savefig(buf, format='png', bbox_inches='tight')
        plt.close(fig)
        buf.seek(0)
        frames.append(imageio.v3.imread(buf))

    imageio.mimsave(output_path, frames, fps=fps, loop=0)
    print(f'Timelapse saved: {output_path}  ({len(frames)} frames @ {fps} fps)')
    return output_path

if 'masked_bands' in vars() and masked_bands:
    from datetime import datetime
    tl_run_id = run_id if 'run_id' in vars() else datetime.utcnow().strftime('%Y%m%d_%H%M%S')
    s3_client = boto3.client(
        's3',
        aws_access_key_id=AWS_ACCESS_KEY_ID,
        aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    )
    for tl_mode, tl_name in [('ndvi', 'ndvi'), ('ndmi', 'ndmi'), ('false_color', 'false_color')]:
        local_path = f'/tmp/timelapse_{tl_name}.gif'
        generate_timelapse(
            masked_bands,
            output_path=local_path,
            mode=tl_mode,
            fps=3,
            start_ym=EVAL_START,
            end_ym=EVAL_END,
        )
        s3_key = f'timelapses/run_id={tl_run_id}/timelapse_{tl_name}.gif'
        s3_client.upload_file(local_path, S3_BUCKET_NAME, s3_key)
        print(f'  Uploaded to s3://{S3_BUCKET_NAME}/{s3_key}')
else:
    print('masked_bands not found. Run the data loading cell first.')


In [ ]:
# Export plot-ready statistics to S3
# Saves: (1) all detected anomalies, (2) full baseline + actual time series for plotting
from datetime import datetime
from pyspark.sql.functions import lit, stddev_pop, mean as spark_mean, when, col

if 'anomaly_df' not in vars() and 'anomaly_pop_df' in vars():
    anomaly_df = anomaly_pop_df  # fall back to population stddev df

if 'S3_BUCKET_NAME' not in vars() or not S3_BUCKET_NAME:
    raise ValueError('S3_BUCKET_NAME is missing.')

run_id = datetime.utcnow().strftime('%Y%m%d_%H%M%S')

# ── 1. Anomaly events (only flagged rows) ─────────────────────────────────────
anomaly_events = anomaly_df.filter(
    col('is_ndvi_anomaly') | col('is_ndmi_anomaly')
).select(
    lit(run_id).alias('run_id'),
    col('true_pixel_id'),
    col('year').cast('int'),
    col('month').cast('int'),
    col('ndvi_mean'),
    col('ndmi_mean'),
    col('ndvi_mu'),
    col('ndvi_sigma'),
    col('ndmi_mu'),
    col('ndmi_sigma'),
    col('ndvi_zscore'),
    col('ndmi_zscore'),
    col('is_ndvi_anomaly'),
    col('is_ndmi_anomaly'),
)

anomaly_path = f's3a://{S3_BUCKET_NAME}/anomaly_events/run_id={run_id}'
anomaly_events.repartition(4).write.mode('overwrite').parquet(anomaly_path)
print(f'Anomaly events written to: {anomaly_path}')
print(f'  Rows: {anomaly_events.count()}')

# ── 2. Full plot-ready time series (all pixels, all months, with baseline) ────
# This is the full anomaly_df — actual observed values + baseline mu/sigma +
# z-scores — needed to plot observed vs baseline band over time.
plot_df = anomaly_df.select(
    lit(run_id).alias('run_id'),
    col('true_pixel_id'),
    col('year').cast('int'),
    col('month').cast('int'),
    col('ndvi_mean'),           # actual observed NDVI
    col('ndmi_mean'),           # actual observed NDMI
    col('ndvi_mu'),             # baseline mean NDVI for this pixel+month
    col('ndvi_sigma'),          # baseline std dev
    col('ndmi_mu'),             # baseline mean NDMI
    col('ndmi_sigma'),
    col('ndvi_zscore'),
    col('ndmi_zscore'),
    col('is_ndvi_anomaly'),
    col('is_ndmi_anomaly'),
)

plot_path = f's3a://{S3_BUCKET_NAME}/plot_stats/run_id={run_id}'
plot_df.repartition(8).write.mode('overwrite').parquet(plot_path)
print(f'Full plot-ready stats written to: {plot_path}')
print(f'  Rows: {plot_df.count()}')

# ── 3. Monthly aggregate (one row per month across ALL pixels) ────────────────
# Great for a region-wide time series chart showing average greenness vs baseline.
from pyspark.sql.functions import avg, sum as spark_sum, count

monthly_agg = anomaly_df.groupBy('year', 'month').agg(
    avg('ndvi_mean').alias('avg_ndvi'),
    avg('ndmi_mean').alias('avg_ndmi'),
    avg('ndvi_mu').alias('avg_ndvi_baseline'),
    avg('ndmi_mu').alias('avg_ndmi_baseline'),
    avg('ndvi_sigma').alias('avg_ndvi_sigma'),
    avg('ndmi_sigma').alias('avg_ndmi_sigma'),
    spark_sum(col('is_ndvi_anomaly').cast('int')).alias('ndvi_anomaly_count'),
    spark_sum(col('is_ndmi_anomaly').cast('int')).alias('ndmi_anomaly_count'),
    count('*').alias('total_pixels'),
).orderBy('year', 'month')

monthly_path = f's3a://{S3_BUCKET_NAME}/monthly_stats/run_id={run_id}'
monthly_agg.repartition(1).write.mode('overwrite').parquet(monthly_path)
print(f'Monthly aggregate stats written to: {monthly_path}')
monthly_agg.show(36)

# ── 4. Pixel coordinate lookup table (true_pixel_id → bbox_wkt) ─────────────
# spark_df still has bbox_wkt before it gets dropped in harmonize_time_series.
# Save once; join with anomaly results later to make spatial maps.
if 'spark_df' not in vars():
    raise ValueError('spark_df is not defined. Re-run the transformation cell first.')

# lon/lat are per-pixel centroids computed from strided_transform in build_pixel_dataframe
pixel_coords = (
    spark_df
    .select('true_pixel_id', 'lon', 'lat')
    .dropDuplicates(['true_pixel_id'])
    .filter(col('lon').isNotNull() & col('lat').isNotNull())
)
coords_path = f's3a://{S3_BUCKET_NAME}/pixel_coords/'
pixel_coords.repartition(4).write.mode('overwrite').parquet(coords_path)
print(f'Pixel coord lookup written to: {coords_path}')
print(f'  Unique pixels: {pixel_coords.count()}')
pixel_coords.show(5)
